# 05 — Multi-Objective AED Location Optimization

**Objective:** Optimize AED placement as a multi-objective facility location problem.

**Four competing objectives:**
1. **Service efficiency** — minimize median distance to nearest AED
2. **Spatial equity** — minimize inter-provincial Gini coefficient
3. **Economic cost** — minimize 10-year total cost (CAPEX + OPEX)
4. **Environmental impact** — minimize lifecycle CO₂ footprint

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.neighbors import BallTree

PROJECT_ROOT = Path('.')
V3_DIR = PROJECT_ROOT / 'mda_project' / 'data' / 'processed_v3'
OUT_DIR = PROJECT_ROOT / 'mda_project' / 'data' / 'output'
OUT_DIR.mkdir(parents=True, exist_ok=True)

mission = pd.read_parquet(V3_DIR / 'mission_records_v3.parquet')
aed = pd.read_parquet(V3_DIR / 'aed_records_v3.parquet')

EARTH_R = 6371.0088

def nearest_km(src, dst):
    """Compute nearest-neighbor distance in km using Haversine."""
    tree = BallTree(np.radians(dst), metric='haversine')
    d, _ = tree.query(np.radians(src), k=1)
    return d[:, 0] * EARTH_R

def gini(x):
    """Compute the Gini coefficient for a distribution."""
    x = np.sort(np.asarray(pd.Series(x).dropna(), dtype=float))
    if len(x) == 0 or x.sum() == 0:
        return 0.0
    n = len(x)
    idx = np.arange(1, n + 1)
    return (2 * (idx * x).sum() / (n * x.sum())) - (n + 1) / n

print(f'Missions: {len(mission):,} | Existing AEDs: {len(aed):,}')

## 5.1 Scenario Evaluation

We evaluate 7 deployment scenarios (10 to 100 new AEDs) using risk-weighted KMeans
for candidate site generation.

In [ ]:
# Composite risk score for weighted candidate generation
base_dist = nearest_km(mission[['latitude','longitude']].to_numpy(),
                       aed[['latitude','longitude']].to_numpy())
mission['base_dist_km'] = base_dist
mission['risk'] = (0.6 * (mission['response_min'] / mission['response_min'].median())
                   + 0.4 * (mission['base_dist_km'] / mission['base_dist_km'].median())).clip(0.5, 6.0)

scenarios = []
for n in [10, 20, 30, 40, 50, 70, 100]:
    km = KMeans(n_clusters=n, random_state=42, n_init=20)
    km.fit(mission[['latitude','longitude']], sample_weight=mission['risk'])
    cand = pd.DataFrame(km.cluster_centers_, columns=['latitude','longitude'])
    cand = cand[cand['latitude'].between(49, 52) & cand['longitude'].between(2, 7)]

    all_aed = pd.concat([aed[['latitude','longitude']], cand], ignore_index=True)
    dist_new = nearest_km(mission[['latitude','longitude']].to_numpy(),
                          all_aed[['latitude','longitude']].to_numpy())

    prov_median = mission.assign(dist_new=dist_new).groupby('province')['dist_new'].median()

    scenarios.append({
        'Scenario': f'S{n}', 'New AEDs': n,
        'Coverage <1km': float((dist_new <= 1.0).mean()),
        'Median Dist [km]': float(np.median(dist_new)),
        'Equity (Gini)': float(gini(prov_median.values)),
        'CAPEX [EUR]': n * 2000,
        'OPEX 10y [EUR]': n * 120 * 10,
        'CO2 [kgCO2e]': n * 85,
    })

scen_df = pd.DataFrame(scenarios)
scen_df['Total Cost [EUR]'] = scen_df['CAPEX [EUR]'] + scen_df['OPEX 10y [EUR]']
print(scen_df)

## 5.2 Pareto Dominance Analysis

In [ ]:
def dominates(a, b):
    """Check if solution a Pareto-dominates solution b (4 objectives)."""
    cond = [a['Median Dist [km]'] <= b['Median Dist [km]'],
            a['Equity (Gini)'] <= b['Equity (Gini)'],
            a['Total Cost [EUR]'] <= b['Total Cost [EUR]'],
            a['CO2 [kgCO2e]'] <= b['CO2 [kgCO2e]']]
    strict = [a['Median Dist [km]'] < b['Median Dist [km]'],
              a['Equity (Gini)'] < b['Equity (Gini)'],
              a['Total Cost [EUR]'] < b['Total Cost [EUR]'],
              a['CO2 [kgCO2e]'] < b['CO2 [kgCO2e]']]
    return all(cond) and any(strict)

pareto_mask = []
for i, a in scen_df.iterrows():
    is_dom = any(dominates(scen_df.iloc[j], a) for j in range(len(scen_df)) if j != i)
    pareto_mask.append(not is_dom)

scen_df['Pareto Optimal'] = pareto_mask
print(scen_df[['Scenario','New AEDs','Coverage <1km','Median Dist [km]','Equity (Gini)','Total Cost [EUR]','Pareto Optimal']])

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

non_pareto = scen_df[~scen_df['Pareto Optimal']]
pareto = scen_df[scen_df['Pareto Optimal']]

ax.scatter(non_pareto['Total Cost [EUR]'], non_pareto['Median Dist [km]'],
           c='#bdc3c7', s=80, label='Dominated', zorder=2)
ax.scatter(pareto['Total Cost [EUR]'], pareto['Median Dist [km]'],
           c='#e74c3c', s=120, edgecolor='black', linewidth=1.5, label='Pareto Optimal', zorder=3)

for _, row in scen_df.iterrows():
    ax.annotate(row['Scenario'], (row['Total Cost [EUR]'], row['Median Dist [km]']),
                textcoords='offset points', xytext=(8, 5), fontsize=10)

ax.set_xlabel('Total Cost (10-year) [EUR]', fontsize=12, fontweight='bold')
ax.set_ylabel('Median Distance to Nearest AED [km]', fontsize=12, fontweight='bold')
ax.set_title('Multi-Objective Pareto Analysis: Cost vs. Coverage', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Summary

The Pareto analysis identifies non-dominated deployment scenarios that represent
the best achievable tradeoffs across efficiency, equity, cost, and environmental impact.

This tabular optimization is complemented by the spatiotemporal deep learning approach
in **Notebook 06** and the advanced Pareto frontier visualization in `05_final_figures.py`.

In [ ]:
# Save scenario table for NB07
(OUT_DIR / 'figures').mkdir(parents=True, exist_ok=True)
scen_out = scen_df.rename(columns={
    'Scenario': 'scenario',
    'New AEDs': 'n_new_aed',
    'Coverage <1km': 'coverage_1km',
    'Median Dist [km]': 'median_dist_km',
    'Equity (Gini)': 'equity_gini',
    'CAPEX [EUR]': 'capex_eur',
    'OPEX 10y [EUR]': 'opex_10y_eur',
    'CO2 [kgCO2e]': 'co2_kgco2e',
    'Total Cost [EUR]': 'total_cost_eur',
    'Pareto Optimal': 'pareto_optimal',
})
scen_out.to_csv(OUT_DIR / 'figures' / 'table_05_scenarios.csv', index=False)
print('Saved table_05_scenarios.csv with', len(scen_out), 'rows')
